# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset with mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Review available record sets, fields, and their `@id`s.

The `mlcroissant` library enables us to list available RecordSets and their nested Fields and Columns. All entities are referenced by their `@id`.


In [ ]:
# Show available RecordSets and their fields/columns by @id

record_sets = metadata.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else 'N/A'}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")
    # Show fields in the recordset
    if hasattr(rs, 'fields') and rs.fields:
        print(f"  Fields and Columns:")
        for f in rs.fields:
            field_id = getattr(f, 'id', None)
            col_id = getattr(f, 'column', None)
            if field_id:
                print(f"    Field @id: {field_id}")
            if col_id is not None:
                # col_id could be a list or a single column
                if isinstance(col_id, list):
                    for col in col_id:
                        print(f"      Column: {getattr(col, 'id', col)}")
                else:
                    print(f"      Column: {getattr(col_id, 'id', col_id)}")
    print('-' * 40)

## 3. Data Extraction

Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we demonstrate loading all record sets into pandas DataFrames, using their `@id` as the dynamic key.

In [ ]:
# Extract data from all discovered record sets
dataframes = {}

# Gather all record_set @id values
record_set_ids = [rs.id for rs in metadata.record_sets]

# Use the first record set for demonstration
main_record_set_id = record_set_ids[0]

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
    except Exception as e:
        print(f"Could not load records for RecordSet @id: {record_set_id}")
        print(e)

if main_record_set_id in dataframes:
    print(f"\nAvailable columns in primary DataFrame ({main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records based on field values, normalizing numeric fields, or grouping for aggregation. We use entity `@id`s for all field references.


In [ ]:
# Choose a numeric field for analysis.
# For demonstration, try to find a numeric column from the first DataFrame.
df = dataframes[main_record_set_id]
numeric_field = None
for col in df.columns:
    # Try to infer a numeric field
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break

if numeric_field is not None:
    # Filtering: select entries where value is above some threshold (here, mean value as example)
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization (Z-score)
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to group by a categorical field (first string/object column not the numeric field)
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].dtype == 'object':
            group_field = col
            break

    if group_field is not None:
        print(f"\nGrouping by {group_field}:\n")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        display(grouped_df.head())
else:
    print("No numeric field found in the main record set for EDA.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset. Here we plot a histogram of the analyzed numeric field, and if grouping was successful, a barplot of grouped means.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If grouped by field above
    if 'grouped_df' in locals() and group_field is not None and not grouped_df.empty:
        plt.figure(figsize=(10, 4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} grouped by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.show()

## 6. Conclusion

In this notebook, we've loaded the FAIR^2 Croissant-formatted mass spectrometry dataset using `mlcroissant`, programmatically discovered available record sets, and performed initial filtering, normalization, and visualization using pandas and seaborn.

**You can now extend the workflow to further explore the dataset's record sets, apply biological/statistical analysis, or build machine learning pipelines. Remember to always use field and record set `@id` references to ensure proper data access and reproducibility.**